In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# ----------------------------
# 1. Загрузка данных
# ----------------------------
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')
test_ids = test['id'].copy()

X = train.drop(columns=['id', 'Heart Disease'])
y = train['Heart Disease']
X_test = test.drop(columns=['id'])

# ----------------------------
# 2. Обработка категориальных признаков
# ----------------------------
# После загрузки X и y
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# Заполнение пропусков
for col in num_cols:
    mean_val = X[col].mean()
    X[col] = X[col].fillna(mean_val)
    X_test[col] = X_test[col].fillna(mean_val)

for col in cat_cols:
    X[col] = X[col].fillna('missing')
    X_test[col] = X_test[col].fillna('missing')

# === КОДИРОВАНИЕ КАТЕГОРИЙ В ЦЕЛЫЕ ЧИСЛА ДЛЯ ВСЕХ МОДЕЛЕЙ ===
label_encoders = {}
X_encoded = X.copy()
X_test_encoded = X_test.copy()

for col in cat_cols:
    le = LabelEncoder()
    # Обучаем на объединённых данных (train + test)
    all_vals = pd.concat([X[col], X_test[col]], axis=0)
    le.fit(all_vals.astype(str))
    X_encoded[col] = le.transform(X[col].astype(str))
    X_test_encoded[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

# Теперь X_encoded и X_test_encoded содержат ТОЛЬКО числовые колонки

# ----------------------------
# 3. Обучение моделей на всём датасете
# ----------------------------

# XGBoost
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
#xgb_model.fit(X_encoded, y)

# LightGBM
lgb_model = lgb.LGBMClassifier(
    objective='binary',
    metric='binary_logloss',
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
#lgb_model.fit(X, y, categorical_feature=cat_cols)

# CatBoost
cat_model = CatBoostClassifier(
    objective='Logloss',
    eval_metric='Logloss',
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=0
)
#cat_model.fit(X, y, cat_features=cat_cols)
# После загрузки y (до обучения моделей)
print("Уникальные значения y:", y.unique())

# Кодируем метки: 'Introvert' → 0, 'Extrovert' → 1
label_map = {'Absence': 0, 'Presence': 1}
y_encoded = y.map(label_map)

# Убедитесь, что все значения корректны
if y_encoded.isna().any():
    raise ValueError("Обнаружены неизвестные метки в y!")

# Кодируем целевую переменную
y_encoded = y.map({'Absence': 0, 'Presence': 1})
if y_encoded.isna().any():
    raise ValueError("Неизвестные метки в y!")

# XGBoost — работает с числами

xgb_model.fit(X_encoded, y_encoded)

# LightGBM — принимает числовые категории

lgb_model.fit(X_encoded, y_encoded, categorical_feature=cat_cols)  # OK, если cat_cols — числа

# CatBoost — может работать со строками, но мы уже закодировали → тоже числа
# Но CatBoost требует указания cat_features как индексы или имена

cat_model.fit(X_encoded, y_encoded, cat_features=cat_cols, verbose=0)
# ----------------------------
# 4. Предсказания на тесте
# ----------------------------
# Предсказания на тесте — ВСЕГДА на закодированных данных!
pred_xgb = xgb_model.predict_proba(X_test_encoded)[:, 1]
pred_lgb = lgb_model.predict_proba(X_test_encoded)[:, 1]
pred_cat = cat_model.predict_proba(X_test_encoded)[:, 1]

# Ансамбль
final_pred = (pred_xgb + pred_lgb + pred_cat) / 3.0
final_class = (final_pred > 0.5).astype(int)

# Обратное преобразование меток
#final_labels = np.where(final_class == 1, 'Presence', 'Absence')

# ----------------------------
# 5. Сохранение сабмита
# ----------------------------


submission = pd.DataFrame({
    'id': test_ids,
    'Heart Disease': final_class  # <-- строковые метки
})
submission.to_csv('ensemble_submission.csv', index=False)
print("Файл ensemble_submission.csv успешно создан!")

Уникальные значения y: ['Presence' 'Absence']
Файл ensemble_submission.csv успешно создан!


***вариант_2***

**вариант 2**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")


In [9]:
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

ID_COL = "id"
TARGET = "Heart Disease"

features = [c for c in train.columns if c not in [ID_COL, TARGET]]

num_features = train[features].select_dtypes(include=["int64", "float64"]).columns
cat_features = train[features].select_dtypes(include=["object"]).columns


In [10]:
def target_encode(
    train_df, test_df, cat_cols, target, n_splits=5, smoothing=10
):
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()

    global_mean = train_df[target].mean()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for col in cat_cols:
        oof = np.zeros(len(train_df))
        test_vals = np.zeros(len(test_df))

        for tr_idx, val_idx in skf.split(train_df, train_df[target]):
            tr, val = train_df.iloc[tr_idx], train_df.iloc[val_idx]

            stats = tr.groupby(col)[target].agg(["mean", "count"])
            smooth = (
                (stats["mean"] * stats["count"] + global_mean * smoothing)
                / (stats["count"] + smoothing)
            )

            oof[val_idx] = val[col].map(smooth).fillna(global_mean)

        stats = train_df.groupby(col)[target].agg(["mean", "count"])
        smooth = (
            (stats["mean"] * stats["count"] + global_mean * smoothing)
            / (stats["count"] + smoothing)
        )

        test_vals = test_df[col].map(smooth).fillna(global_mean)

        train_encoded[col + "_te"] = oof
        test_encoded[col + "_te"] = test_vals

    return train_encoded, test_encoded


In [19]:
def fast_target_encoding(
    train, test, cat_cols, target, n_splits=5, smoothing=10
):
    train = train.copy()
    test = test.copy()

    global_mean = train[target].mean()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for col in cat_cols:
        oof = np.zeros(len(train))

        for tr_idx, val_idx in skf.split(train, train[target]):
            tr = train.iloc[tr_idx]
            val = train.iloc[val_idx]

            stats = (
                tr.groupby(col)[target]
                .agg(["mean", "count"])
                .reset_index()
            )

            stats["te"] = (
                stats["mean"] * stats["count"]
                + global_mean * smoothing
            ) / (stats["count"] + smoothing)

            val = val[[col]].merge(
                stats[[col, "te"]],
                on=col,
                how="left"
            )

            oof[val_idx] = val["te"].fillna(global_mean).values

        train[col + "_te"] = oof

        # test
        stats = (
            train.groupby(col)[target]
            .agg(["mean", "count"])
            .reset_index()
        )

        stats["te"] = (
            stats["mean"] * stats["count"]
            + global_mean * smoothing
        ) / (stats["count"] + smoothing)

        test[col + "_te"] = (
            test[[col]]
            .merge(stats[[col, "te"]], on=col, how="left")["te"]
            .fillna(global_mean)
            .values
        )

    return train, test


In [ ]:
train_te, test_te = fast_target_encoding(
    train, test, cat_features, TARGET
)


In [14]:
train_fe = train_te.drop(columns=cat_features)
test_fe  = test_te.drop(columns=cat_features)


In [15]:
corr = train[num_features + [TARGET]].corr()[TARGET].abs().sort_values(ascending=False)
top_nums = corr.index[1:6]

for i in range(len(top_nums)):
    for j in range(i + 1, len(top_nums)):
        f1, f2 = top_nums[i], top_nums[j]
        train_fe[f"{f1}_x_{f2}"] = train_fe[f1] * train_fe[f2]
        test_fe[f"{f1}_x_{f2}"]  = test_fe[f1] * test_fe[f2]


KeyError: "None of [Index(['AgeHeart Disease', 'SexHeart Disease', 'Chest pain typeHeart Disease',\n       'BPHeart Disease', 'CholesterolHeart Disease',\n       'FBS over 120Heart Disease', 'EKG resultsHeart Disease',\n       'Max HRHeart Disease', 'Exercise anginaHeart Disease',\n       'ST depressionHeart Disease', 'Slope of STHeart Disease',\n       'Number of vessels fluroHeart Disease', 'ThalliumHeart Disease'],\n      dtype='object')] are in the [columns]"

In [16]:
for col in top_nums:
    train_fe[f"{col}_sq"] = train_fe[col] ** 2
    test_fe[f"{col}_sq"]  = test_fe[col] ** 2


NameError: name 'top_nums' is not defined

In [ ]:
X = train_fe.drop(columns=[ID_COL, TARGET])
y = train_fe[TARGET]

X_test = test_fe.drop(columns=[ID_COL])


In [ ]:
lgb_params = dict(
    n_estimators=1200,
    learning_rate=0.02,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=50,
    random_state=42
)


In [ ]:
def oof_lgb(X, y, X_test):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        model = LGBMClassifier(**lgb_params)

        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
            eval_metric="auc",
            verbose=100
        )

        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / 5

        print(f"Fold {fold} AUC:", roc_auc_score(y.iloc[val_idx], oof[val_idx]))

    print("LGB Mean AUC:", roc_auc_score(y, oof))
    return oof, test_preds


In [ ]:
lgb_oof, lgb_test = oof_lgb(X, y, X_test)


In [ ]:
cat_model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.02,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200
)


In [ ]:
cat_oof = np.zeros(len(X))
cat_test = np.zeros(len(X_test))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    cat_model.fit(
        X.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
        use_best_model=True
    )

    cat_oof[val_idx] = cat_model.predict_proba(X.iloc[val_idx])[:, 1]
    cat_test += cat_model.predict_proba(X_test)[:, 1] / 5

print("CatBoost AUC:", roc_auc_score(y, cat_oof))


In [ ]:
oof_blend = 0.6 * lgb_oof + 0.4 * cat_oof
print("Blend AUC:", roc_auc_score(y, oof_blend))


In [ ]:
test_blend = 0.6 * lgb_test + 0.4 * cat_test


submission = pd.DataFrame({
    "id": test[ID_COL],
    "Heart Disease": test_blend
})

submission.to_csv("submission.csv", index=False)
submission.head()


вариант 3

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

In [3]:
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

ID_COL = "id"
TARGET = "Heart Disease"

features = [c for c in train.columns if c not in [ID_COL, TARGET]]

num_features = train[features].select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = train[features].select_dtypes(include=["object"]).columns.tolist()

In [5]:
for col in cat_features:
    freq = train[col].value_counts(normalize=True)
    train[col + "_freq"] = train[col].map(freq)
    test[col + "_freq"]  = test[col].map(freq).fillna(0)

In [13]:
train

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,1
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,0
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,0
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,0
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,629995,56,0,1,110,226,0,0,132,0,0.0,1,0,7,0
629996,629996,54,1,4,128,249,1,2,150,0,0.0,2,0,3,0
629997,629997,67,1,4,130,275,0,0,149,0,0.0,1,2,7,1
629998,629998,52,1,4,140,199,0,2,157,0,0.0,1,0,6,1


In [14]:
corr = train[num_features + [TARGET]].corr()[TARGET].abs().sort_values(ascending=False)

top_nums = corr.index[1:6].tolist()

for i in range(len(top_nums)):
    for j in range(i + 1, len(top_nums)):
        f1, f2 = top_nums[i], top_nums[j]
        train[f"{f1}_x_{f2}"] = train[f1] * train[f2]
        test[f"{f1}_x_{f2}"]  = test[f1] * test[f2]

In [15]:
for col in top_nums:
    train[f"{col}_sq"] = train[col] ** 2
    test[f"{col}_sq"]  = test[col] ** 2

In [16]:
lgb_features = (
    num_features
    + [c + "_freq" for c in cat_features]
    + [c for c in train.columns if "_x_" in c or c.endswith("_sq")]
)

X = train[lgb_features]
y = train[TARGET]
X_test = test[lgb_features]

In [17]:
lgb_params = dict(
    n_estimators=1000,
    learning_rate=0.02,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=50,
    random_state=42
)

In [19]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_oof = np.zeros(len(train))
lgb_test = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    model = LGBMClassifier(**lgb_params)

    model.fit(
        X.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
        eval_metric="auc"
    )

    lgb_oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    lgb_test += model.predict_proba(X_test)[:, 1] / 5

    print(f"Fold {fold} AUC:",
          roc_auc_score(y.iloc[val_idx], lgb_oof[val_idx]))

print("LGB AUC:", roc_auc_score(y, lgb_oof))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.057262 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1321
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
Fold 0 AUC: 0.9553370714508596
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.055391 

In [20]:
cb_features = num_features + cat_features

X_cb = train[cb_features]
X_cb_test = test[cb_features]

cat_idx = [X_cb.columns.get_loc(c) for c in cat_features]

In [21]:
cat_oof = np.zeros(len(train))
cat_test = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cb, y)):
    model = CatBoostClassifier(
        iterations=2500,
        learning_rate=0.03,
        depth=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=200
    )

    model.fit(
        X_cb.iloc[tr_idx], y.iloc[tr_idx],
        cat_features=cat_idx,
        eval_set=(X_cb.iloc[val_idx], y.iloc[val_idx]),
        use_best_model=True
    )

    cat_oof[val_idx] = model.predict_proba(X_cb.iloc[val_idx])[:, 1]
    cat_test += model.predict_proba(X_cb_test)[:, 1] / 5

print("CatBoost AUC:", roc_auc_score(y, cat_oof))

0:	test: 0.9466095	best: 0.9466095 (0)	total: 126ms	remaining: 5m 14s
200:	test: 0.9547250	best: 0.9547250 (200)	total: 12.5s	remaining: 2m 23s
400:	test: 0.9550991	best: 0.9550991 (400)	total: 24.3s	remaining: 2m 7s
600:	test: 0.9554484	best: 0.9554484 (600)	total: 36.1s	remaining: 1m 54s
800:	test: 0.9555838	best: 0.9555838 (800)	total: 48s	remaining: 1m 41s
1000:	test: 0.9556351	best: 0.9556363 (994)	total: 1m	remaining: 1m 30s
1200:	test: 0.9556402	best: 0.9556481 (1067)	total: 1m 12s	remaining: 1m 18s
1400:	test: 0.9556287	best: 0.9556481 (1067)	total: 1m 24s	remaining: 1m 6s
1600:	test: 0.9556033	best: 0.9556481 (1067)	total: 1m 36s	remaining: 54.1s
1800:	test: 0.9555792	best: 0.9556481 (1067)	total: 1m 48s	remaining: 42s
2000:	test: 0.9555422	best: 0.9556481 (1067)	total: 2m	remaining: 30s
2200:	test: 0.9555137	best: 0.9556481 (1067)	total: 2m 12s	remaining: 18s
2400:	test: 0.9554808	best: 0.9556481 (1067)	total: 2m 24s	remaining: 5.95s
2499:	test: 0.9554559	best: 0.9556481 (106

In [22]:
oof_blend = 0.5 * lgb_oof + 0.5 * cat_oof
print("Blend AUC:", roc_auc_score(y, oof_blend))

test_blend = 0.6 * lgb_test + 0.4 * cat_test

Blend AUC: 0.9552508373012216


In [24]:
d = (test_blend > 0.5).astype(int)

In [25]:
submission = pd.DataFrame({
    "id": test[ID_COL],
    "Heart Disease": d
})

submission.to_csv("submission.csv", index=False)
submission.head()

,id,Heart Disease
0,630000,1
1,630001,0
2,630002,1
3,630003,0
4,630004,0
